# 迁移学习 Transfer Learning

- 基本步骤：
    - 加载预训练模型；
    - 冻结特征提取层；
    - 替换分类头；
    - 微调新分类头，只更新新添加的分类层的参数；
- 策略取决于目标数据量：很少（几百），仅替换最后一层分类头，冻结其他层所有参数，较小学习率；适中（几千），解冻预训练模型靠近输出端的最后几层；较大（几万），替换分类头并允许整个网络的参数进行更新，使用稍大学习率或采用学习率衰减策略；
- 为什么主要替换最后一层：分类需求不同；特征组合不同；特征通用性，浅层和中间层学习的特征比较基础，是通用的

在pytorch上进行迁移学习

In [5]:
from langchain_community.embeddings.dashscope import BATCH_SIZE
from torchvision.models import resnet18, ResNet18_Weights
import torch
import torch.nn as nn

In [6]:
### 只更换分类头

model = resnet18(weights=ResNet18_Weights.DEFAULT)    # 加载 ImageNet 预训练权重
for param in model.parameters():
    param.requires_grad = False         # 冻结所有层

in_features = model.fc.in_features      # 提取模型中fc层的输入维度
model.fc = nn.Linear(in_features, 1)    # 新生成一个二分类头

In [8]:
### 更换分类头，同时解冻第四阶段的参数

model = resnet18(weights=ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False         # 冻结所有层

# 解冻第四阶段的参数（最后一个残差连接块）
for param in model.layer4.parameters():
    param.requires_grad = True

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 1)    # 新生成一个二分类头

应用 ResNet 和 迁移学习

In [9]:
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random
import torch
import torch.nn as nn
from torchvision import transforms, models
import os

In [10]:
def verify_images(image_folder):
    classes = ["Cat", "Dog"]
    class_to_idx = {"Cat": 0, "Dog": 1}
    samples = []
    for cls_name in classes:
        cls_dir = os.path.join(image_folder, cls_name)
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            path = os.path.join(cls_dir, fname)
            try:
                with Image.open(path) as img:
                    img.verify()
                samples.append((path, class_to_idx[cls_name]))
            except Exception:
                print(f"Warning: Skipping corrupted image {path}")
    return samples


class ImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        with Image.open(path) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, label


def evaluate(model, test_dataloader):
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in test_dataloader:
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            outputs = torch.sigmoid(model(inputs))
            preds = (outputs > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    return val_acc


if __name__ == "__main__":
    DATA_DIR = r"D:\agent\torch\data\archive\PetImages"
    BATCH_SIZE = 64
    IMG_SIZE = 128
    EPOCHS = 15
    LR = 0.001
    PRINT_STEP = 100
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    all_samples = verify_images(DATA_DIR)
    random.seed(42)
    random.shuffle(all_samples)
    train_size = int(len(all_samples) * 0.8)
    train_sample = all_samples[:train_size]
    valid_sample = all_samples[train_size:]

    train_transform = transforms.Compose([
        transforms.Resize((150,150)),
        transforms.RandomCrop(size=(IMG_SIZE,IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1
        ),
        transforms.RandomRotation(30),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225]),
    ])

    train_dataset = ImageDataset(train_sample, train_transform)
    valid_dataset = ImageDataset(valid_sample, valid_transform)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False     # 冻结所有层

    # 解冻第四阶段的参数（最后一个残差块）
    for param in model.layer4.parameters():
        param.requires_grad = True

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)    # 二分类
    model = model.to(DEVICE)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        model.train()
        running_loss = 0.0

        for step, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            optimizer.zero_grad()
            outputs = torch.sigmoid(model(inputs))
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if (step + 1) % PRINT_STEP == 0:
                avg_loss = running_loss / PRINT_STEP
                print(f"  Step {step + 1}, Loss: {avg_loss:.4f}")
                running_loss = 0.0

        valid_acc = evaluate(model, valid_loader)
        print(f"Validation Accuracy after epoch {epoch+1}: {valid_acc:.4f}")

C:\Users\61075\miniforge3\envs\rag\lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1/15
  Step 100, Loss: 0.2416
  Step 200, Loss: 0.1828
  Step 300, Loss: 0.1660
Validation Accuracy after epoch 1: 0.9520
Epoch 2/15
  Step 100, Loss: 0.1367
  Step 200, Loss: 0.1533
  Step 300, Loss: 0.1511
Validation Accuracy after epoch 2: 0.9590
Epoch 3/15
  Step 100, Loss: 0.1291
  Step 200, Loss: 0.1268
  Step 300, Loss: 0.1201
Validation Accuracy after epoch 3: 0.9630
Epoch 4/15
  Step 100, Loss: 0.1181
  Step 200, Loss: 0.1281
  Step 300, Loss: 0.1226
Validation Accuracy after epoch 4: 0.9606
Epoch 5/15
  Step 100, Loss: 0.1165
  Step 200, Loss: 0.1090
  Step 300, Loss: 0.1079
Validation Accuracy after epoch 5: 0.9620
Epoch 6/15
  Step 100, Loss: 0.1073
  Step 200, Loss: 0.1024
  Step 300, Loss: 0.1045
Validation Accuracy after epoch 6: 0.9642
Epoch 7/15
  Step 100, Loss: 0.0954
  Step 200, Loss: 0.1000
  Step 300, Loss: 0.0971
Validation Accuracy after epoch 7: 0.9656
Epoch 8/15
  Step 100, Loss: 0.0860
  Step 200, Loss: 0.0913
  Step 300, Loss: 0.0904
Validation Accurac